In [1]:
import pandas as pd
import numpy as np

# Menampilkan angka desimal dengan pemisah ribuan agar tidak muncul notasi ilmiah (mis. 1.256382e+08)
pd.options.display.float_format = '{:,.2f}'.format

df = pd.read_csv('paket_a_transaksi_pos_2024_2025.csv')

# Pembersihan Data

In [2]:
# Merapikan kode_produk (spasi di tepi dan huruf kecil)
df['kode_produk'] = df['kode_produk'].str.strip().str.upper()

# Menyeragamkan format tanggal (yyyy-mm-dd dan dd/mm/yyyy atau dd-mm-yyyy)
tgl_iso = pd.to_datetime(df['tanggal'], format='%Y-%m-%d', errors='coerce')
tgl_dmy = pd.to_datetime(df['tanggal'].str.replace('/', '-', regex=False), format='%d-%m-%Y', errors='coerce')
df['tanggal'] = tgl_iso.fillna(tgl_dmy)

print('Baris data mentah:', len(df))

Baris data mentah: 5030


In [3]:
# Menghapus baris duplikat
print('Duplikat dihapus:', df.duplicated().sum())
df = df.drop_duplicates()

# Menghapus kode produk tidak valid (awalan XXX)
print('Kode produk XXX dihapus:', df['kode_produk'].str.startswith('XXX').sum())
df = df[~df['kode_produk'].str.startswith('XXX')]

# Menghapus outlier jumlah (nilai wajar maksimal 12 unit, outlier bernilai ratusan hingga ribuan)
print('Outlier jumlah dihapus:', (df['jumlah'].abs() > 100).sum())
df = df[df['jumlah'].abs() <= 100].reset_index(drop=True)

print('Baris data bersih:', len(df))

Duplikat dihapus: 30
Kode produk XXX dihapus: 55
Outlier jumlah dihapus: 12
Baris data bersih: 4933


In [4]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 4933 entries, 0 to 4932
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id_transaksi   4933 non-null   str           
 1   tanggal        4933 non-null   datetime64[us]
 2   kode_gerai     4933 non-null   str           
 3   kode_kasir     4675 non-null   str           
 4   kode_produk    4933 non-null   str           
 5   jumlah         4933 non-null   int64         
 6   harga_satuan   4933 non-null   str           
 7   diskon_persen  4933 non-null   int64         
 8   status         4933 non-null   str           
dtypes: datetime64[us](1), int64(2), str(6)
memory usage: 347.0 KB


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,2025-03-26,G05,K016,RMT1407,1,6000,0,SELESAI


# Task 1 — Indexing & Selection (Pembersihan Anomali & Imputasi)

**1. Pembersihan Format Harga**

In [5]:
# Tipe data harga_satuan sebelum dibersihkan (terbaca sebagai objek/teks)
print(df['harga_satuan'].dtype)
print(df.loc[~df['harga_satuan'].str.fullmatch(r'\d+'), 'harga_satuan'].unique()[:10])

str
<StringArray>
[ '7500,00', '11000,00',   '13.000', '27000,00', '13500,00',   '28.000',
 '43500,00', '16500,00',  '4500,00',   '11.000']
Length: 10, dtype: str


In [6]:
# Nilai seperti '13.000' memakai titik sebagai pemisah ribuan, bukan desimal,
# sehingga titiknya dihapus terlebih dahulu agar tidak terbaca 13.0
ribuan = df['harga_satuan'].str.fullmatch(r'\d{1,3}\.\d{3}')
df.loc[ribuan, 'harga_satuan'] = df.loc[ribuan, 'harga_satuan'].str.replace('.', '', regex=False)

# Mengubah koma desimal menjadi titik (contoh: '7500,00' -> '7500.00'), lalu konversi ke float
df['harga_satuan'] = df['harga_satuan'].str.replace(',', '.', regex=False).astype(float)

print(df['harga_satuan'].dtype)
df['harga_satuan'].describe()

float64


count    4,933.00
mean    22,149.10
std     15,188.92
min      4,000.00
25%     11,000.00
50%     16,000.00
75%     32,000.00
max     54,000.00
Name: harga_satuan, dtype: float64

**2. Penanganan Kuantitas Negatif**

In [7]:
print('Jumlah negatif sebelum:', (df['jumlah'] < 0).sum())

# Boolean indexing dengan .loc[] untuk mengubah jumlah negatif menjadi positif
df.loc[df['jumlah'] < 0, 'jumlah'] = df.loc[df['jumlah'] < 0, 'jumlah'].abs()

print('Jumlah negatif sesudah:', (df['jumlah'] < 0).sum())

Jumlah negatif sebelum: 39
Jumlah negatif sesudah: 0


**3. Imputasi Nilai Kosong Kasir**

In [8]:
print('NaN kode_kasir sebelum:', df['kode_kasir'].isna().sum())

# Mengisi nilai kosong kode_kasir dengan label default
df.loc[df['kode_kasir'].isna(), 'kode_kasir'] = 'K_UNKNOWN'

print('NaN kode_kasir sesudah:', df['kode_kasir'].isna().sum())

NaN kode_kasir sebelum: 258
NaN kode_kasir sesudah: 0


**4. Seleksi Posisi Slicing**

In [9]:
# 5 baris pertama dengan kolom id_transaksi, tanggal, kode_gerai, dan jumlah
df.iloc[:5, [0, 1, 2, 5]]

,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,2025-03-26,G05,1


# Task 2 — Operasi Aritmatika (Feature Engineering)

**1. Perhitungan Omzet Kotor**

In [10]:
df['Total_Kotor'] = df['harga_satuan'] * df['jumlah']
df[['id_transaksi', 'harga_satuan', 'jumlah', 'Total_Kotor']].head()

,id_transaksi,harga_satuan,jumlah,Total_Kotor
0,TRX0241065,"13,000.00",4,"52,000.00"
1,TRX0240510,"21,000.00",1,"21,000.00"
2,TRX0244483,"7,500.00",3,"22,500.00"
3,TRX0242532,"6,500.00",2,"13,000.00"
4,TRX0243509,"6,000.00",1,"6,000.00"


**2. Perhitungan Diskon**

In [11]:
# diskon_persen berskala 0-100, sehingga dibagi 100 terlebih dahulu
df['Nominal_Diskon'] = df['Total_Kotor'] * (df['diskon_persen'] / 100)
df[['id_transaksi', 'Total_Kotor', 'diskon_persen', 'Nominal_Diskon']].head()

,id_transaksi,Total_Kotor,diskon_persen,Nominal_Diskon
0,TRX0241065,"52,000.00",0,0.00
1,TRX0240510,"21,000.00",0,0.00
2,TRX0244483,"22,500.00",0,0.00
3,TRX0242532,"13,000.00",0,0.00
4,TRX0243509,"6,000.00",0,0.00


**3. Perhitungan Omzet Bersih & Pajak**

In [12]:
# (Total_Kotor - Nominal_Diskon) * 1.11  -> sudah termasuk PPN 11%
df['Total_Bersih'] = (df['Total_Kotor'] - df['Nominal_Diskon']) * 1.11
df[['id_transaksi', 'Total_Kotor', 'Nominal_Diskon', 'Total_Bersih']].head()

,id_transaksi,Total_Kotor,Nominal_Diskon,Total_Bersih
0,TRX0241065,"52,000.00",0.00,"57,720.00"
1,TRX0240510,"21,000.00",0.00,"23,310.00"
2,TRX0244483,"22,500.00",0.00,"24,975.00"
3,TRX0242532,"13,000.00",0.00,"14,430.00"
4,TRX0243509,"6,000.00",0.00,"6,660.00"


# Task 3 — Filtering (Penyaringan Data)

**1. Filter Multi-Kondisi**

In [13]:
filter_multi = df[
    (df['status'] == 'SELESAI') &
    (df['kode_gerai'].isin(['G05', 'G09'])) &
    (df['Total_Bersih'] > 100000)
]

print('Jumlah baris yang memenuhi kriteria:', len(filter_multi))
filter_multi

Jumlah baris yang memenuhi kriteria: 134


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
7,TRX0241251,2024-10-09,G09,K035,SGR1204,4,"47,500.00",10,SELESAI,"190,000.00","19,000.00","189,810.00"
24,TRX0244253,2024-07-18,G09,K026,BYI1502,4,"30,500.00",0,SELESAI,"122,000.00",0.00,"135,420.00"
25,TRX0244999,2024-08-09,G05,K015,MIN1003,4,"28,500.00",0,SELESAI,"114,000.00",0.00,"126,540.00"
89,TRX0240898,2025-12-09,G05,K010,BYI1509,3,"46,500.00",0,SELESAI,"139,500.00",0.00,"154,845.00"
204,TRX0241868,2025-07-01,G05,K009,SGR1203,6,"20,500.00",0,SELESAI,"123,000.00",0.00,"136,530.00"
...,...,...,...,...,...,...,...,...,...,...,...,...
4716,TRX0244151,2025-04-25,G09,K013,RMT1408,4,"52,500.00",5,SELESAI,"210,000.00","10,500.00","221,445.00"
4734,TRX0244705,2025-08-22,G09,K020,BYI1506,4,"29,000.00",0,SELESAI,"116,000.00",0.00,"128,760.00"
4787,TRX0243898,2024-04-23,G05,K008,SGR1204,12,"51,000.00",0,SELESAI,"612,000.00",0.00,"679,320.00"
4820,TRX0241966,2024-04-22,G09,K024,SGR1206,3,"42,000.00",0,SELESAI,"126,000.00",0.00,"139,860.00"


**2. Filter dengan Query**

In [14]:
filter_g10 = df.query('kode_gerai == "G10" and jumlah >= 5')
filter_g10

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
83,TRX0243880,2024-08-04,G10,K002,MKN1101,12,"45,500.00",0,SELESAI,"546,000.00",0.00,"606,060.00"
164,TRX0241592,2024-05-11,G10,K033,SGR1202,6,"13,000.00",0,SELESAI,"78,000.00",0.00,"86,580.00"
347,TRX0242686,2024-02-03,G10,K016,BYI1502,12,"26,000.00",0,SELESAI,"312,000.00",0.00,"346,320.00"
545,TRX0241396,2025-12-16,G10,K009,RMT1406,12,"5,500.00",5,SELESAI,"66,000.00","3,300.00","69,597.00"
862,TRX0243559,2025-09-30,G10,K016,SGR1206,6,"47,000.00",0,SELESAI,"282,000.00",0.00,"313,020.00"
946,TRX0244939,2025-07-08,G10,K009,SGR1200,6,"52,000.00",10,SELESAI,"312,000.00","31,200.00","311,688.00"
970,TRX0243076,2025-02-18,G10,K003,MKN1100,6,"18,000.00",0,SELESAI,"108,000.00",0.00,"119,880.00"
1002,TRX0242437,2025-09-18,G10,K030,MKN1104,6,"6,000.00",0,SELESAI,"36,000.00",0.00,"39,960.00"
1233,TRX0240537,2025-05-09,G10,K027,BYI1509,6,"54,000.00",0,SELESAI,"324,000.00",0.00,"359,640.00"
1269,TRX0240698,2024-01-24,G10,K022,MKN1109,6,"29,000.00",0,SELESAI,"174,000.00",0.00,"193,140.00"


# Task 4 — Sorting (Pengurutan Data)

**1. Multi-Column Sorting**

In [15]:
df_sorted = df.sort_values(by=['kode_gerai', 'Total_Bersih'], ascending=[True, False])
df_sorted

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
2900,TRX0241367,2025-05-15,G01,K006,RMT1408,12,"50,500.00",0,SELESAI,"606,000.00",0.00,"672,660.00"
3517,TRX0240608,2025-07-25,G01,K006,MKN1101,12,"47,000.00",0,SELESAI,"564,000.00",0.00,"626,040.00"
2881,TRX0240631,2024-12-24,G01,K013,RMT1408,12,"42,000.00",0,SELESAI,"504,000.00",0.00,"559,440.00"
5,TRX0240658,2025-07-18,G01,K008,RMT1408,12,"45,500.00",10,SELESAI,"546,000.00","54,600.00","545,454.00"
3815,TRX0241537,2024-10-19,G01,K013,MKN1102,12,"27,000.00",0,SELESAI,"324,000.00",0.00,"359,640.00"
...,...,...,...,...,...,...,...,...,...,...,...,...
3652,TRX0242116,2025-10-31,G12,K024,MIN1008,1,"4,500.00",0,SELESAI,"4,500.00",0.00,"4,995.00"
1455,TRX0241390,2024-12-29,G12,K008,PRW1300,1,"5,500.00",20,SELESAI,"5,500.00","1,100.00","4,884.00"
2251,TRX0244124,2024-01-26,G12,K035,MIN1008,1,"5,500.00",20,SELESAI,"5,500.00","1,100.00","4,884.00"
3776,TRX0242420,2024-01-29,G12,K019,RMT1406,1,"4,000.00",0,SELESAI,"4,000.00",0.00,"4,440.00"


**2. Top-N Records**

In [16]:
top_5_transaksi = df.nlargest(5, 'Total_Bersih')
top_5_transaksi

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
3665,TRX0242177,2025-06-01,G07,K038,BYI1509,12,"54,000.00",0,SELESAI,"648,000.00",0.00,"719,280.00"
1188,TRX0241211,2024-01-28,G02,K017,MIN1006,12,"52,500.00",0,SELESAI,"630,000.00",0.00,"699,300.00"
630,TRX0243320,2024-04-14,G09,K034,RMT1408,12,"52,000.00",0,SELESAI,"624,000.00",0.00,"692,640.00"
4787,TRX0243898,2024-04-23,G05,K008,SGR1204,12,"51,000.00",0,SELESAI,"612,000.00",0.00,"679,320.00"
2900,TRX0241367,2025-05-15,G01,K006,RMT1408,12,"50,500.00",0,SELESAI,"606,000.00",0.00,"672,660.00"


# Task 5 — Grouping & Agregasi (Pelaporan Kinerja Gerai)

**1. Laporan Agregasi Per Gerai**

In [17]:
# Menyaring hanya transaksi berstatus SELESAI
df_selesai = df[df['status'] == 'SELESAI']

laporan_gerai = df_selesai.groupby('kode_gerai', as_index=False).agg(
    Jumlah_Transaksi=('id_transaksi', 'count'),
    Total_Unit_Terjual=('jumlah', 'sum'),
    Total_Omzet_Bersih=('Total_Bersih', 'sum'),
    Rata_Rata_Transaksi=('Total_Bersih', 'mean')
).sort_values(by='Total_Omzet_Bersih', ascending=False).round(2)

laporan_gerai

,kode_gerai,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
0,G01,439,1160,"28,428,126.75","64,756.55"
7,G08,402,1048,"27,241,148.25","67,764.05"
6,G07,373,990,"25,069,516.50","67,210.50"
10,G11,391,1063,"24,918,168.00","63,729.33"
9,G10,398,991,"24,543,792.75","61,667.82"
3,G04,394,1034,"24,458,933.25","62,078.51"
11,G12,390,980,"24,003,888.75","61,548.43"
1,G02,374,977,"23,219,174.25","62,083.35"
4,G05,385,1024,"23,101,264.50","60,003.28"
8,G09,368,931,"22,105,511.25","60,069.32"


**2. Transformasi Kelompok (Window Function)**

In [18]:
# (Total_Bersih / Total_Omzet_Gerai_Tersebut) * 100
total_omzet_gerai = df.groupby('kode_gerai')['Total_Bersih'].transform('sum')
df['Kontribusi_Gerai_Persen'] = (df['Total_Bersih'] / total_omzet_gerai) * 100

# Kontribusi per transaksi nilainya kecil, sehingga ditampilkan dengan 6 desimal
with pd.option_context('display.float_format', '{:,.6f}'.format):
    display(df[['id_transaksi', 'kode_gerai', 'Total_Bersih', 'Kontribusi_Gerai_Persen']].head())

# Pemeriksaan: total kontribusi tiap gerai harus tepat 100%
df.groupby('kode_gerai')['Kontribusi_Gerai_Persen'].sum().round(2)

,id_transaksi,kode_gerai,Total_Bersih,Kontribusi_Gerai_Persen
0,TRX0241065,G09,"57,720.000000",0.245786
1,TRX0240510,G09,"23,310.000000",0.099260
2,TRX0244483,G11,"24,975.000000",0.094246
3,TRX0242532,G05,"14,430.000000",0.060697
4,TRX0243509,G05,"6,660.000000",0.028014


kode_gerai
G01   100.00
G02   100.00
G03   100.00
G04   100.00
G05   100.00
G06   100.00
G07   100.00
G08   100.00
G09   100.00
G10   100.00
G11   100.00
G12   100.00
Name: Kontribusi_Gerai_Persen, dtype: float64